# IMPORT

In [ ]:
import pandas as pd
from pathlib import Path
import joblib
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# DEFINE FOLDERS

In [ ]:
STATION_CLUSTER_FOLDER = Path("station_clusters")
DATASET_INPUT_FOLDER = Path("correlated_data_csv")
MODEL_OUTPUT_FOLDER = Path("cluster_models")
PLOT_FOLDER = Path("model_plots")

folders = [STATION_CLUSTER_FOLDER, DATASET_INPUT_FOLDER, MODEL_OUTPUT_FOLDER, PLOT_FOLDER]
for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

# LOAD DATA

In [ ]:
df = pd.read_csv(f"{DATASET_INPUT_FOLDER}/final_clean_data.csv")
clusters = pd.read_csv(f"{STATION_CLUSTER_FOLDER}/station_clusters_2_42.csv")

df = df.merge(clusters, on="station_code", how="inner")

print("Rows:", df.shape)
print("Clusters:", df["cluster"].value_counts())
print(df.columns)

# LOAD MODELS

In [ ]:
models = {}
for i in range(2):
    models[i] = joblib.load(f"{MODEL_OUTPUT_FOLDER}/model_cluster_{i}.pkl")

# FEATURE SELECTION

In [ ]:
FEATURES = [
    # Time
    "sin_hour",
    "cos_hour",
    "is_weekend",

    # Weather
    "temperature",
    "wind_speed",
    "wind_direction",
    "weather_code",
    "is_day",

    # Station context
    "capacity",
]

TARGET = "total_bikes_now"


# QUICK TEST ON THE MODELS

In [ ]:
test_clusters = sorted(df["cluster"].unique())

for cluster_id in test_clusters:
    df_c = df[df["cluster"] == cluster_id].sort_values("timestamp")
    
    X = df_c[FEATURES]
    y = df_c[TARGET]
    
    model_name = models[cluster_id]
    if model_name == "LinearRegression":
        model = LinearRegression()
    elif model_name == "SGDRegressor":
        model = Pipeline([("scaler", StandardScaler()),
                          ("sgd", SGDRegressor(max_iter=2000, tol=1e-3))])
    else:
        model = RandomForestRegressor(n_estimators=200, max_depth=15,
                                      random_state=42, n_jobs=-1)
    
    model = models[cluster_id]
    preds = model.predict(X)
    
    plt.figure(figsize=(12,4))
    plt.plot(df_c["timestamp"], y, label="Actual", alpha=0.7)
    plt.plot(df_c["timestamp"], preds, label="Predicted", alpha=0.7)
    plt.title(f"Cluster {cluster_id} - {model_name}")
    plt.xlabel("Timestamp")
    plt.ylabel("Total Bikes Now")
    plt.legend()
    plt.savefig(PLOT_FOLDER / f"cluster_{cluster_id}_{model_name}.png")
    plt.close()

C:\Users\emirh\AppData\Local\Temp\ipykernel_43420\1190566852.py:33: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(PLOT_FOLDER / f"cluster_{cluster_id}_{model_name}.png")
C:\Users\emirh\AppData\Local\Temp\ipykernel_43420\1190566852.py:33: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(PLOT_FOLDER / f"cluster_{cluster_id}_{model_name}.png")


In [ ]:
for cluster_id in sorted(clusters_df["cluster"].unique()):
    print(f"Plotting cluster {cluster_id}")

    # 1️⃣ Get all stations in this cluster
    stations_in_cluster = clusters_df[
        clusters_df["cluster"] == cluster_id
    ]["station_code"].unique()

    # 2️⃣ Pick ONE random station
    station_code = np.random.choice(stations_in_cluster)

    # 3️⃣ Filter data for this station only
    df_station = (
        df[df["station_code"] == station_code]
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    if len(df_station) < 100:
        print(f"  Skipped station {station_code} (too few samples)")
        continue

    X = df_station[FEATURES]
    y = df_station[TARGET]

    # 4️⃣ Load trained model (DO NOT refit)
    model_path = MODEL_OUTPUT_FOLDER / f"model_cluster_{cluster_id}.pkl"
    model = joblib.load(model_path)

    preds = model.predict(X)

    # 5️⃣ Plot
    plt.figure(figsize=(14, 5))
    plt.plot(df_station["timestamp"], y, label="Actual", alpha=0.8)
    plt.plot(df_station["timestamp"], preds, label="Predicted", alpha=0.8)

    plt.title(
        f"Cluster {cluster_id} | Station {station_code}\n"
        f"Model: {type(model).__name__}"
    )
    plt.xlabel("Timestamp")
    plt.ylabel("Total Bikes Now")
    plt.legend()
    plt.tight_layout()

    # 6️⃣ Save
    plt.savefig(
        PLOT_FOLDER / f"cluster_{cluster_id}_station_{station_code}.png"
    )
    plt.close()